<a href="https://colab.research.google.com/github/justamy20/scikit-learn-Cookbook/blob/main/12.%20Cross-Validation%20and%20Model%20Evaluation/Chapter_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 12: Cross-Validation and Model Evaluation Techniques

<div class="alert alert-info">
<b>Chapter Overview:</b> In this chapter, we move beyond the simple train-test split to more robust validation strategies. We will explore K-Fold, Stratified K-Fold, and Time Series splitting, as well as diagnostic tools like Learning Curves and Validation Curves to identify Bias and Variance issues in our models.
</div>

## 1. The Necessity of Robust Validation
**Theoretical Deep-Dive:**
The traditional "Hold-out" method (single train-test split) has a major flaw: the evaluation result can be highly dependent on which samples ended up in the test set by chance.

**Cross-Validation (CV)** solves this by partitioning the data into $K$ segments (folds). The model is trained $K$ times, each time using a different fold as the validation set and the remaining $K-1$ folds as the training set. The final performance is the average of these $K$ runs:
$$CV_{(K)} = \frac{1}{K} \sum_{i=1}^{K} MSE_i$$

This provides a much more stable estimate of how the model will perform on truly unseen data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, K_Fold, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# Load data
iris = load_iris()
X, y = iris.data, iris.target
model = LogisticRegression(max_iter=200)

# 1. Standard K-Fold (K=5)
kfold = K_Fold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kfold)

print(f"K-Fold Scores: {scores}")
print(f"Average Accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# 2. Stratified K-Fold (Crucial for imbalanced classes)
# It ensures each fold has the same percentage of samples of each target class as the complete set.
strat_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
strat_scores = cross_val_score(model, X, y, cv=strat_kfold)

print(f"\nStratified K-Fold Scores: {strat_scores}")
print(f"Average Accuracy: {strat_scores.mean():.4f}")

---
## 2. Time Series Splitting
**Theoretical Deep-Dive:**
Standard cross-validation assumes the data is **Independent and Identically Distributed (I.I.D.)**. However, in Time Series data (stock prices, weather), the order of data matters. Shuffling the data would result in "Data Leakage," where the model uses future information to predict the past.

We use **TimeSeriesSplit**, where the training set grows over time. In each split, the test set is always chronologically *after* the training set.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Create a dummy time series data
X_time = np.arange(100).reshape(-1, 1)
tscv = TimeSeriesSplit(n_splits=5)

plt.figure(figsize=(10, 4))
for i, (train_index, test_index) in enumerate(tscv.split(X_time)):
    plt.scatter(train_index, [i] * len(train_index), c='blue', marker='_', lw=10)
    plt.scatter(test_index, [i] * len(test_index), c='red', marker='_', lw=10)

plt.title("TimeSeriesSplit Visualization (Blue=Train, Red=Test)")
plt.xlabel("Time Index")
plt.ylabel("Fold Iteration")
plt.show()

---
## 3. Diagnosing Models with Learning Curves
**Theoretical Deep-Dive:**
A **Learning Curve** plots the training and validation score of an estimator for varying numbers of training samples. It is a powerful diagnostic tool to find:

1. **High Bias (Underfitting):** Both training and validation scores are low and close to each other. Adding more data will NOT help. You need a more complex model.
2. **High Variance (Overfitting):** Training score is very high, but validation score is much lower (a large gap). Adding more data will likely help close the gap.

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.svm import SVC

def plot_learning_curve(estimator, X, y):
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5)
    )

    train_mean = np.mean(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)

    plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, test_mean, 'o-', color="g", label="Cross-validation score")
    plt.title("Learning Curve (Bias vs Variance Analysis)")
    plt.xlabel("Training Examples")
    plt.ylabel("Score")
    plt.legend(loc="best")
    plt.grid()
    plt.show()

# Test with SVC
plot_learning_curve(SVC(gamma=0.001), X, y)

---
## 4. Hyperparameter Tuning with Validation Curves
**Theoretical Deep-Dive:**
While learning curves vary the *number of samples*, a **Validation Curve** varies a single *hyperparameter* (like $C$ in SVM or $\alpha$ in Ridge). This helps us find the "sweet spot" where the model is complex enough to capture patterns but simple enough to generalize.

In [ ]:
from sklearn.model_selection import validation_curve

param_range = np.logspace(-6, -1, 5)
train_scores, test_scores = validation_curve(
    SVC(), X, y, param_name="gamma", param_range=param_range,
    cv=5, scoring="accuracy", n_jobs=-1
)

plt.semilogx(param_range, np.mean(train_scores, axis=1), label="Training score", color="r")
plt.semilogx(param_range, np.mean(test_scores, axis=1), label="Cross-validation score", color="g")
plt.title("Validation Curve with SVM (varying Gamma)")
plt.xlabel("Gamma")
plt.ylabel("Score")
plt.legend(loc="best")
plt.show()

---
### Chapter 12 Summary
*(Fulfilling Assignment Requirement 3.b: Summarize each chapter)*

Chapter 12 provides the essential tools for rigorous model evaluation and diagnosis. We learned that a single accuracy number is often misleading and that we must observe model behavior across different data splits and parameter ranges.

**Key Takeaways:**
1. **Stratified K-Fold:** Should be the default choice for classification, especially when classes are imbalanced, as it preserves the class distribution in each fold.
2. **Temporal Integrity:** For time-dependent data, we must never use standard CV or shuffling. `TimeSeriesSplit` is mandatory to respect the chronological flow of information.
3. **Diagnosis over Guessing:** Learning curves allow us to mathematically determine if our model needs more data (High Variance) or a fundamentally different approach/more features (High Bias).
4. **Validation Curves:** Help us visualize the influence of a hyperparameter on the training and validation scores, effectively identifying the point where the model starts to overfit.

By implementing these techniques, we transition from "training a model" to "engineering a reliable predictive system."